# Paso 7 · Unión de estado y demanda

En este notebook unimos los dos parquets generados en pasos anteriores:

- `ocupacion_2022.parquet` → estado por estación y hora (ocupación, capacidad, coordenadas)
- `viajes_2022.parquet` → demanda por estación y hora (salidas, entradas)

El resultado es `estado_demanda_2022.parquet`, una única tabla con todo lo necesario para empezar a construir el dataset de entrenamiento del modelo.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_ocupacion = pd.read_parquet("ocupacion_2022.parquet")
df_viajes    = pd.read_parquet("viajes_2022.parquet")

print(f"Ocupación: {len(df_ocupacion):,} filas | columnas: {list(df_ocupacion.columns)}")
print(f"Viajes:    {len(df_viajes):,} filas | columnas: {list(df_viajes.columns)}")

Ocupación: 2,233,046 filas | columnas: ['_id', 'name', 'total_bases', 'free_bases', 'longitude', 'latitude', 'dock_bikes', 'id', 'ocupation']
Viajes:    1,625,077 filas | columnas: ['id', 'hora', 'salidas', 'entradas']


In [3]:
# La columna temporal se llama distinto en cada tabla: armonizar
df_ocupacion = df_ocupacion.rename(columns={"_id": "hora"})

# El _id de la ocupación tiene segundos (ej. 2022-01-01 00:13:20), redondeamos a la hora
df_ocupacion["hora"] = pd.to_datetime(df_ocupacion["hora"]).dt.floor("h")

# Aseguramos tipos consistentes en la clave id
df_ocupacion["id"] = df_ocupacion["id"].astype(int)
df_viajes["id"]    = df_viajes["id"].astype(int)

print("Rango de ocupación:", df_ocupacion["hora"].min(), "->", df_ocupacion["hora"].max())
print("Rango de viajes:   ", df_viajes["hora"].min(),    "->", df_viajes["hora"].max())

Rango de ocupación: 2022-01-01 00:00:00 -> 2022-12-31 23:00:00
Rango de viajes:    2022-01-01 00:00:00 -> 2022-12-31 23:00:00


In [4]:
df_completo = pd.merge(
    df_ocupacion,
    df_viajes,
    on=["id", "hora"],
    how="left"
)

# Las celdas sin viajes registrados → 0
df_completo[["salidas", "entradas"]] = df_completo[["salidas", "entradas"]].fillna(0).astype(int)

print(f"Filas tras la unión: {len(df_completo):,}")
print(f"Columnas finales:    {list(df_completo.columns)}")
df_completo.head()

Filas tras la unión: 2,233,046
Columnas finales:    ['hora', 'name', 'total_bases', 'free_bases', 'longitude', 'latitude', 'dock_bikes', 'id', 'ocupation', 'salidas', 'entradas']


,hora,name,total_bases,free_bases,longitude,latitude,dock_bikes,id,ocupation,salidas,entradas
0,2022-01-01,Miguel Moya,24,16,-3.705842,40.420589,7,3,0.304348,0,0
1,2022-01-01,Plaza Conde Suchil,18,1,-3.706917,40.430294,14,4,0.933333,0,1
2,2022-01-01,Malasaña,24,2,-3.702587,40.428552,17,5,0.894737,0,0
3,2022-01-01,Fuencarral,27,14,-3.702050,40.428520,10,6,0.416667,0,1
4,2022-01-01,Colegio Arquitectos,24,11,-3.698447,40.424148,10,7,0.476190,0,1


In [5]:
# 1) ¿Cuáles son las filas duplicadas? Mirar un ejemplo
duplicados = df_completo[df_completo.duplicated(subset=['id','hora'], keep=False)]
print(f"Total filas implicadas en duplicados: {len(duplicados):,}")
print(f"\nEjemplo de duplicados (mismas id y hora):")
print(duplicados.sort_values(['id','hora']).head(10))

# 2) ¿En qué meses se concentran?
print(f"\nDistribución de duplicados por mes:")
print(duplicados['hora'].dt.month.value_counts().sort_index())

# 3) ¿Qué horas únicas hay realmente vs esperadas?
horas_unicas = df_completo['hora'].drop_duplicates().sort_values()
print(f"\nHoras únicas: {len(horas_unicas)}")
print(f"Primera: {horas_unicas.iloc[0]}, Última: {horas_unicas.iloc[-1]}")
# ¿Hay huecos?
diferencias = horas_unicas.diff().dropna()
print(f"Saltos > 1 hora: {(diferencias > pd.Timedelta('1h')).sum()}")

Total filas implicadas en duplicados: 25,193

Ejemplo de duplicados (mismas id y hora):
                      hora              name  total_bases  free_bases  \
460217 2022-03-16 17:00:00  Puerta del Sol A           30          18   
460476 2022-03-16 17:00:00  Puerta del Sol A           30          18   
462811 2022-03-17 02:00:00  Puerta del Sol A           30          21   
463071 2022-03-17 02:00:00  Puerta del Sol A           30          21   
463331 2022-03-17 03:00:00  Puerta del Sol A           30          21   
463591 2022-03-17 03:00:00  Puerta del Sol A           30          21   
464371 2022-03-17 06:00:00  Puerta del Sol A           30          20   
464631 2022-03-17 06:00:00  Puerta del Sol A           30          20   
464891 2022-03-17 07:00:00  Puerta del Sol A           30          20   
465151 2022-03-17 07:00:00  Puerta del Sol A           30          20   

        longitude   latitude  dock_bikes  id  ocupation  salidas  entradas  
460217  -3.701834  40.417214   

In [6]:
print("=" * 60)
print("ESTRUCTURA")
print("=" * 60)
print(f"Filas:             {len(df_completo):,}")
print(f"Columnas:          {len(df_completo.columns)} → {list(df_completo.columns)}")
print(f"Estaciones únicas: {df_completo['id'].nunique()}")
print(f"Rango temporal:    {df_completo['hora'].min()}  →  {df_completo['hora'].max()}")
print(f"Horas únicas:      {df_completo['hora'].nunique()}  (esperado ≈ 8.760)")
print(f"Duplicados (id+hora): {df_completo.duplicated(subset=['id','hora']).sum()}  (debe ser 0)")

print("\n" + "=" * 60)
print("INTEGRIDAD DEL MERGE")
print("=" * 60)
print(f"Filas idénticas a ocupación: {len(df_completo) == 2_233_046}  (debe ser True)")
print(f"Total salidas en el merge:  {df_completo['salidas'].sum():,}")
print(f"Total entradas en el merge: {df_completo['entradas'].sum():,}")
print(f"Total salidas en df_viajes: {df_viajes['salidas'].sum():,}  (debería ser >= al del merge)")
print(f"Total entradas en df_viajes:{df_viajes['entradas'].sum():,}  (debería ser >= al del merge)")
print(f"  → Si los del merge son menores, son viajes 'huérfanos' en horas con estación caída")

print("\n" + "=" * 60)
print("CALIDAD DE LAS COLUMNAS CLAVE")
print("=" * 60)
print(f"{'Columna':18s} {'NaN':>10s} {'% NaN':>8s} {'Mín':>10s} {'Máx':>10s}")
for c in ["ocupation", "dock_bikes", "free_bases", "total_bases", "salidas", "entradas"]:
    n = df_completo[c].isna().sum()
    pct = n / len(df_completo) * 100
    print(f"{c:18s} {n:>10,} {pct:>7.3f}% {df_completo[c].min():>10.2f} {df_completo[c].max():>10.2f}")

print("\n" + "=" * 60)
print("COHERENCIA SEMÁNTICA")
print("=" * 60)
print(f"Ocupación fuera de [0,1]:           {((df_completo['ocupation'] < 0) | (df_completo['ocupation'] > 1)).sum()}  (debe ser 0)")
print(f"Salidas negativas:                  {(df_completo['salidas'] < 0).sum()}  (debe ser 0)")
print(f"Entradas negativas:                 {(df_completo['entradas'] < 0).sum()}  (debe ser 0)")
print(f"Filas con dock_bikes > total_bases: {(df_completo['dock_bikes'] > df_completo['total_bases']).sum()}  (debe ser 0)")
print(f"Filas con free_bases > total_bases: {(df_completo['free_bases'] > df_completo['total_bases']).sum()}  (debe ser 0)")

print("\n" + "=" * 60)
print("DISTRIBUCIÓN DE LA DEMANDA")
print("=" * 60)
print(f"Filas con (salidas=0 y entradas=0): {((df_completo['salidas']==0) & (df_completo['entradas']==0)).sum():,} ({((df_completo['salidas']==0) & (df_completo['entradas']==0)).mean()*100:.1f}%)")
print(f"Filas con algún movimiento:         {((df_completo['salidas']>0) | (df_completo['entradas']>0)).sum():,} ({((df_completo['salidas']>0) | (df_completo['entradas']>0)).mean()*100:.1f}%)")

ESTRUCTURA
Filas:             2,233,046
Columnas:          11 → ['hora', 'name', 'total_bases', 'free_bases', 'longitude', 'latitude', 'dock_bikes', 'id', 'ocupation', 'salidas', 'entradas']
Estaciones únicas: 263
Rango temporal:    2022-01-01 00:00:00  →  2022-12-31 23:00:00
Horas únicas:      8687  (esperado ≈ 8.760)
Duplicados (id+hora): 12723  (debe ser 0)

INTEGRIDAD DEL MERGE
Filas idénticas a ocupación: True  (debe ser True)
Total salidas en el merge:  3,541,253
Total entradas en el merge: 3,537,015
Total salidas en df_viajes: 3,544,128  (debería ser >= al del merge)
Total entradas en df_viajes:3,539,751  (debería ser >= al del merge)
  → Si los del merge son menores, son viajes 'huérfanos' en horas con estación caída

CALIDAD DE LAS COLUMNAS CLAVE
Columna                   NaN    % NaN        Mín        Máx
ocupation                 804   0.036%       0.00       1.00
dock_bikes                  0   0.000%       0.00      30.00
free_bases                  0   0.000%       0.00  

In [7]:
print(f"Antes: {len(df_completo):,} filas")

# Como los duplicados son idénticos, da lo mismo quedarse con el primero o el último
df_completo = df_completo.drop_duplicates(subset=['id', 'hora'], keep='first').reset_index(drop=True)

print(f"Después: {len(df_completo):,} filas")
print(f"Eliminados: {2_233_046 - len(df_completo):,} duplicados")

# Re-comprobación final
print(f"\nDuplicados restantes: {df_completo.duplicated(subset=['id','hora']).sum()}")
print(f"Horas únicas: {df_completo['hora'].nunique()}")

Antes: 2,233,046 filas
Después: 2,220,323 filas
Eliminados: 12,723 duplicados

Duplicados restantes: 0
Horas únicas: 8687


In [8]:
df_completo.to_parquet("estado_demanda_2022.parquet", index=False)
print(f"✓ Guardado: estado_demanda_2022.parquet")
print(f"  Filas:     {len(df_completo):,}")
print(f"  Columnas:  {len(df_completo.columns)}")
print(f"  Periodo:   {df_completo['hora'].min()}  →  {df_completo['hora'].max()}")
print(f"  Estaciones: {df_completo['id'].nunique()}")

✓ Guardado: estado_demanda_2022.parquet
  Filas:     2,220,323
  Columnas:  11
  Periodo:   2022-01-01 00:00:00  →  2022-12-31 23:00:00
  Estaciones: 263
